# Advanced Debugging & Diagnostics Guide

**Master deep-dive debugging techniques, log analysis, profiling, and diagnostic procedures for complex production issues.**

- Structured debugging methodology
- Log analysis & correlation
- Profiling & performance diagnostics
- Memory leak detection
- Database query analysis
- Network debugging
- Distributed tracing
- Production incident diagnosis

**Estimated time:** 60 minutes | **Skill level:** Advanced | **For:** Senior engineers, DevOps, troubleshooters


## 1. Structured Debugging Methodology

### The Scientific Debugging Method

```markdown
Problem: "The payment API returns 500 errors sporadically"

Step 1: Reproduce
├─ Can you reproduce consistently? (No)
├─ How often: 1 in 100 requests
├─ Under what conditions: High load only
└─ Last occurrence: 14:32 UTC yesterday

Step 2: Isolate
├─ Is it always the same endpoint? (Yes)
├─ Same user? (No, varies)
├─ Same payment provider? (Yes, Stripe)
├─ Same error message? (Yes, 500)

Step 3: Hypothesis
Theory 1: Rate limiting hit (Stripe limit exceeded)
Theory 2: Database connection pool exhausted
Theory 3: Timeout on Stripe API call

Step 4: Test
├─ Check Stripe rate limits: /logs show we're at 95% quota
├─ Check DB pool: Max 20 connections, 19 in use during spike
├─ Check timeout: Stripe calls average 800ms, timeout is 1s

Step 5: Verdict
ROOT CAUSE: Stripe rate limit + DB pool saturation combined

- High payment volume → hits Stripe limit
- Retries consume DB connections
- Pool exhausted → new requests wait
- Timeout triggers → 500 error

Step 6: Fix

1. Increase Stripe rate limit (contact Stripe)
2. Increase DB pool from 20 → 30
3. Implement exponential backoff retry
4. Add circuit breaker for Stripe
5. Test under load
```

### Question-Driven Debugging

```markdown
When stuck, ask:

1. What changed recently?
    - Deployment in last 24h? (Yes/No)
    - Config change? (Yes/No)
    - Traffic pattern change? (Yes/No)
2. Is it environmental?
    - Happens in prod only? (Yes/No)
    - Happens in staging? (Yes/No)
    - Happens locally? (Yes/No)
3. Is it load-related?
    - Happens at high load? (Yes/No)
    - Happens at low load? (Yes/No)
    - Concurrent? (Yes/No)
4. Is it deterministic?
    - Always happens? (Yes/No)
    - Intermittent? (Yes/No)
    - Random? (Yes/No)
5. What's the scope?
    - All users? (Yes/No)
    - Specific user? (Yes/No)
    - Specific region? (Yes/No)
    - Specific data? (Yes/No)
```

### Debugging Toolbox

```markdown
| Tool            | Purpose               | When to Use          |
| --------------- | --------------------- | -------------------- |
| Logs            | What happened?        | Always first         |
| Traces          | Request flow          | Complex request path |
| Metrics         | When did it change?   | Performance issues   |
| Profiler        | Where's the time?     | Slow endpoint        |
| Debugger        | Step through code     | Logic errors         |
| Load tester     | Can I reproduce?      | Intermittent issues  |
| Network sniffer | What's in the packet? | Network issues       |
```


## 2. Log Analysis & Correlation

### Structured Logging Format

```json
{
    "timestamp": "2024-12-17T14:32:45.123Z",
    "level": "ERROR",
    "logger": "PaymentService",
    "correlation_id": "req-a1b2c3d4e5f6",
    "request_id": "stripe-webhook-789",
    "user_id": "user-123",
    "service": "payment-api",
    "host": "prod-payment-02",

    "message": "Failed to process payment",
    "error": {
        "type": "StripeRateLimitError",
        "message": "Too many requests",
        "code": "rate_limit_exceeded"
    },

    "context": {
        "provider": "stripe",
        "amount": 4999,
        "currency": "USD",
        "retry_count": 3,
        "retry_delay_ms": 1000
    },

    "performance": {
        "duration_ms": 850,
        "db_queries": 2,
        "external_calls": 1
    },

    "stack_trace": [
        "File payment_service.py line 234 in process_payment",
        "File stripe_client.py line 145 in create_charge"
    ]
}
```

### Log Search Patterns

```markdown
# Find all errors in payment service (last hour)

logs:
service: "payment-api"
level: "ERROR"
time_range: "-1h"

Result: 34 errors in last hour

# Drill down: Which endpoint?

filter:
endpoint: "/api/payment/checkout"

Result: 28 errors on checkout endpoint

# Which user?

filter:
user_id: "user-123"

Result: 8 errors from this user

# What's the pattern?

correlation_id: "req-a1b2c3d4e5f6"

Result: Complete request trace with all services

1. API Gateway receives request (0ms)
2. Auth service validates (45ms)
3. Payment service called (850ms) ← SLOW
4. Response returned (851ms)
```

### Correlation ID Tracing

```python
# All services log correlation_id to track single request

# API Gateway
def handle_request(request):
    correlation_id = request.headers.get('X-Correlation-ID') or uuid4()
    logger.info("Request received", extra={
        'correlation_id': correlation_id
    })
    return call_payment_service(correlation_id)

# Payment Service
def process_payment(correlation_id, payment_data):
    logger.info("Payment processing started", extra={
        'correlation_id': correlation_id
    })

    try:
        result = stripe.create_charge(
            amount=payment_data.amount,
            metadata={'correlation_id': correlation_id}
        )
    except StripeError as e:
        logger.error("Stripe error", extra={
            'correlation_id': correlation_id,
            'error': str(e)
        })

# Now query by correlation_id finds ALL related logs
logs | where correlation_id == "req-a1b2c3d4e5f6"

Results:
- API Gateway: Request received
- Auth service: Token validated
- Payment service: Processing started
- Stripe client: API call made
- Stripe error: Rate limited
- Payment service: Error logged
- API Gateway: 500 response sent
```

### Common Log Patterns

```markdown
Error Pattern: "Connection pool exhausted"
Logs show:
├─ 19 connections in use
├─ New request waits
├─ 30 second timeout
└─ Returns error

Root cause: Slow database queries holding connections

Pattern: Cascading failures
Logs show:
├─ Service A times out (20s)
├─ Service B retries (3x20s = 60s)
├─ Service C backs off (exponential)
└─ Everything stops

Root cause: Upstream service degraded

Pattern: Intermittent errors
Logs show:
├─ Error then success then error
├─ No correlation with load
├─ Random timing
└─ Different errors each time

Root cause: Network flakiness or timing issue
```


## 3. Profiling & Performance Diagnostics

### Python Profiling Tools

#### cProfile: CPU Time Profiling

```python
import cProfile
import pstats

# Profile a function
profiler = cProfile.Profile()
profiler.enable()

# Run the code
process_payment(100, "stripe")

profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats('cumulative').print_stats(10)

# Output:
#   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
#     1000   0.015   0.000    2.850   0.003 payment_service.py:45(validate_card)
#     1000   0.020   0.000    1.200   0.001 stripe_client.py:78(create_charge)
#       50   0.001   0.000    0.950   0.019 database.py:120(query)
```

#### line_profiler: Line-by-Line Timing

```python
from line_profiler import LineProfiler

lp = LineProfiler()
lp_wrapper = lp(process_payment)
lp_wrapper(100, "stripe")
lp.print_stats()

# Output:
# Timer unit: 1e-06 s
#
#     Line #      Hits         Time  Per Hit   % Time  Line Contents
#     45            1        100.0    100.0     0.5  @profile
#     46         1000      50000.0     50.0    20.5  for payment in payments:
#     47         1000      15000.0     15.0     6.2      validate(payment)
#     48         1000      80000.0     80.0    32.8      stripe.charge(payment)  ← SLOW
#     49         1000      60000.0     60.0    24.6      log_transaction(payment)
```

#### memory_profiler: Memory Leaks

```python
from memory_profiler import profile

@profile
def load_training_data(size_mb):
    data = []
    for i in range(size_mb):
        # Each iteration adds ~1MB
        data.append(bytearray(1024 * 1024))

    # Memory should peak at size_mb MB
    # Then return to baseline when function ends
    return len(data)

# Run: python -m memory_profiler train.py
# Output shows memory at each line
```

### Performance Analysis Workflow

```markdown
Step 1: Establish baseline
├─ Measure endpoint: ~200ms
├─ Measure database query: ~50ms
├─ Measure external API: ~100ms
└─ Total: ~350ms

Step 2: Identify slow part
├─ Profile request
├─ Results show: Validation loop (150ms)
├─ That's 40% of total time
└─ Target for optimization

Step 3: Understand why
├─ cProfile shows validation calls stripe API 50x
├─ Should only call once
├─ Loop should validate locally first

Step 4: Optimize
├─ Move stripe check outside loop
├─ Now validation: 5ms (50x faster!)
├─ Total request: ~155ms (55% improvement)

Step 5: Verify
├─ Profile again after change
├─ Confirm new baseline: 155ms
├─ Monitor in production
```


## 4. Memory Leak Detection

### Identifying Memory Leaks

```python
import tracemalloc

# Start tracking
tracemalloc.start()

# Run code that might leak
for i in range(1000):
    process_payment(i)

# Get snapshot
current, peak = tracemalloc.get_traced_memory()
print(f"Current: {current / 1024 / 1024:.1f}MB")
print(f"Peak: {peak / 1024 / 1024:.1f}MB")

# Expected:
# If no leak: Current ≈ 10MB (baseline)
# If leak: Current ≈ 500MB (keeps growing)
```

### Common Memory Leak Patterns

```markdown
Pattern 1: Global Cache Without Limit
❌ Bad:
cache = {} # Grows forever

def get_user(user_id):
if user_id not in cache:
cache[user_id] = fetch_from_db(user_id)
return cache[user_id]

✅ Good:
from functools import lru_cache

@lru_cache(maxsize=1000)
def get_user(user_id):
return fetch_from_db(user_id)
```

```markdown
Pattern 2: Event Listeners Not Removed
❌ Bad:
def subscribe_to_events():
client.on('payment.complete', handle_payment) # Never unsubscribed

# Called many times → many listeners accumulate

✅ Good:
def subscribe_to_events():
handler = lambda: handle_payment()
client.on('payment.complete', handler)
return handler # Return so we can unsubscribe later

def cleanup():
client.off('payment.complete', handler)
```

```markdown
Pattern 3: Circular References in Classes
❌ Bad:
class Node:
def **init**(self, value):
self.value = value
self.parent = None
self.children = []

a = Node(1)
b = Node(2)
a.children.append(b)
b.parent = a # Circular reference
del a, b # Not freed (circular reference prevents garbage collection)

✅ Good:
import weakref

class Node:
def **init**(self, value):
self.value = value
self.parent = None # Use weakref
self.children = []

a = Node(1)
b = Node(2)
a.children.append(b)
b.parent = weakref.ref(a) # Weak reference
del a, b # Now properly freed
```


## 5. Database Query Analysis

### Query Performance Investigation

```sql
-- Find slow queries (MySQL/PostgreSQL)
SELECT query_time, lock_time, query
FROM mysql.slow_log
WHERE query_time > 1.0  -- Queries over 1 second
ORDER BY query_time DESC
LIMIT 10;
```

### Query Optimization Process

```markdown
Step 1: Identify slow query
SELECT \* FROM transactions
WHERE user_id = 123
AND created_at > '2024-01-01'
AND status = 'completed';
-- Execution time: 5 seconds ❌

Step 2: Analyze execution plan
EXPLAIN SELECT ...;

Result:
├─ Seq Scan on transactions (full table scan)
├─ Filters applied: user_id, created_at, status
├─ Rows examined: 10,000,000
└─ Problem: No index on (user_id, created_at, status)

Step 3: Add index
CREATE INDEX idx_user_transaction_status
ON transactions(user_id, created_at, status);

Step 4: Rerun query
-- Execution time: 15ms ✅ (333x faster!)

Step 5: Verify
EXPLAIN SELECT ...;
├─ Index Scan idx_user_transaction_status
├─ Rows examined: 150 (vs 10M before)
└─ Perfect!
```

### Connection Pool Monitoring

```markdown
Symptoms: "Database connection pool exhausted"

Diagnosis:
├─ Check current connections:
│ SELECT COUNT(_) FROM information_schema.processlist;
│ Result: 19/20 (95% full)
│
├─ Check duration of connections:
│ SELECT _ FROM processlist
│ WHERE time > 300; -- > 5 minutes
│ Result: 3 queries running for 30 minutes!
│
├─ Kill long-running queries:
│ KILL QUERY process_id;
│
└─ Result: Pool drops to 5/20 (normal)

Prevention:
├─ Set statement timeout (5 minutes)
├─ Increase pool size from 20 → 30
├─ Add alerts at 80% capacity
└─ Daily monitoring of long queries
```


## 6. Network Debugging

### Packet Inspection

```bash
# Capture Stripe API calls
tcpdump -i any host api.stripe.com -w stripe_capture.pcap

# Analyze with Wireshark
wireshark stripe_capture.pcap

# Or with tshark
tshark -r stripe_capture.pcap -Y http

# Output:
# 1   0.000   client → stripe   TLS: Client Hello
# 2   0.050   stripe → client   TLS: Server Hello
# 3   0.051   client → stripe   HTTP: POST /v1/charges
# 4   0.200   stripe → client   HTTP: 200 OK
```

### Network Latency Analysis

```python
import time
import requests

def measure_api_latency():
    endpoints = [
        'https://api.stripe.com/v1/charges',
        'https://api.twilio.com/messages',
        'https://sql-db.azure.com:1433'
    ]

    for endpoint in endpoints:
        start = time.time()
        try:
            response = requests.get(endpoint, timeout=5)
            latency_ms = (time.time() - start) * 1000
            print(f"{endpoint}: {latency_ms:.0f}ms")
        except requests.Timeout:
            print(f"{endpoint}: Timeout (>5000ms)")

# Output:
# https://api.stripe.com/v1/charges: 145ms
# https://api.twilio.com/messages: 200ms
# https://sql-db.azure.com:1433: 50ms (local region)
```

### DNS Resolution Issues

```bash
# Check DNS resolution time
time dig api.stripe.com

# Output:
# Query time: 45 ms  ← This should be < 100ms
# SERVER: 8.8.8.8#53(8.8.8.8)

# If slow, try different DNS
time dig @1.1.1.1 api.stripe.com
time dig @8.8.8.8 api.stripe.com

# If one is fast, update resolv.conf
echo "nameserver 1.1.1.1" | sudo tee /etc/resolv.conf
```


## 7. Distributed Tracing

### OpenTelemetry Tracing

```python
from opentelemetry import trace
from opentelemetry.exporter.jaeger import JaegerExporter
from opentelemetry.sdk.trace import TracerProvider

# Setup tracer
jaeger_exporter = JaegerExporter(agent_host_name="localhost", agent_port=6831)
trace.set_tracer_provider(TracerProvider())
trace.get_tracer_provider().add_span_processor(...)

tracer = trace.get_tracer(__name__)

# Create spans for each operation
def process_payment(payment_id):
    with tracer.start_as_current_span("process_payment") as span:
        span.set_attribute("payment_id", payment_id)

        # Validate
        with tracer.start_as_current_span("validate_payment") as v_span:
            v_span.set_attribute("validation_type", "card")
            validate_payment(payment_id)

        # Call Stripe
        with tracer.start_as_current_span("stripe_charge") as s_span:
            s_span.set_attribute("provider", "stripe")
            stripe_result = stripe.create_charge(payment_id)

        # Log to database
        with tracer.start_as_current_span("log_transaction") as l_span:
            l_span.set_attribute("duration_ms", 150)
            log_transaction(payment_id, stripe_result)

# Trace visualization in Jaeger:
# process_payment (160ms)
# ├─ validate_payment (10ms)
# ├─ stripe_charge (100ms)
# │  └─ Network request (80ms)
# └─ log_transaction (40ms)
```

### Using Traces to Find Issues

```markdown
Trace shows: stripe_charge takes 5 seconds
├─ Should take 100ms normally
├─ Something is slow

Investigation:
├─ Look at network spans: 4900ms
├─ That's the API call to Stripe
├─ Stripe is responding slowly

Action:
├─ Check Stripe status page: Down
├─ Implementation: Add timeout + fallback
├─ Result: Prevent cascade failures
```


## 8. Production Incident Diagnosis

### Incident Response Checklist

```markdown
Alert: "Payment API returning 500 errors"

[ ] Immediate (0-5 minutes)
[ ] Confirm the issue
[ ] Check status page / dashboards
[ ] Severity level (P1/P2/P3?)
[ ] Impact scope (% users affected?)

[ ] Initial Assessment (5-15 minutes)
[ ] Check recent deployments
[ ] Check infrastructure metrics
[ ] Check error logs for root cause clue
[ ] Check external dependencies status

[ ] Diagnosis (15-60 minutes)
[ ] Profile the error (how often, when started?)
[ ] Check logs + traces for correlation
[ ] Check metrics (CPU, memory, connections)
[ ] Hypothesis: What changed?

[ ] Temporary Mitigation (if needed, before fix)
[ ] Can we degrade gracefully? (Cache, fallback)
[ ] Can we route around the issue? (Circuit breaker)
[ ] Can we throttle load? (Rate limiting)

[ ] Root Cause Fix
[ ] Implement fix
[ ] Test in staging
[ ] Deploy to production with canary (10% traffic)
[ ] Monitor metrics and errors

[ ] Verification
[ ] Error rate back to normal
[ ] Performance back to baseline
[ ] No new errors in logs
[ ] Users reporting successful payments

[ ] Post-Incident
[ ] Root cause analysis
[ ] Process improvements
[ ] Documentation updates
[ ] Team learning session
```

### Diagnostic Quick Reference

```markdown
Symptom: "500 errors"
├─ Check: Error logs for exception
├─ Check: External API status
├─ Check: Database connections
├─ Check: Memory usage
└─ Most common: Database pool exhausted

Symptom: "Timeouts"
├─ Check: Downstream service latency
├─ Check: Database query performance
├─ Check: Network latency to external APIs
├─ Check: CPU usage (process context switch overhead)
└─ Most common: Downstream service slow

Symptom: "Intermittent failures"
├─ Check: Network flakiness (packet loss?)
├─ Check: DNS resolution
├─ Check: Load-based issues (fails under spike)
├─ Check: Race conditions in code
└─ Most common: Network or concurrency

Symptom: "Memory growing"
├─ Check: Cache size unchecked
├─ Check: Event listeners not unsubscribed
├─ Check: Circular references
├─ Check: Large result sets loaded to memory
└─ Most common: Unbounded cache
```


## Advanced Debugging Checklist

### When Stuck

- [ ] Recreate the issue (reproduce = half solved)
- [ ] Isolate the scope (narrow down surface area)
- [ ] Check logs first (80% of answers in logs)
- [ ] Profile if performance related
- [ ] Trace if request path is unclear
- [ ] Ask "what changed?" (most issues are recent changes)
- [ ] Test hypothesis (don't guess)
- [ ] Document findings (helps next time)

### Before Giving Up

- [ ] Searched GitHub issues / Stack Overflow?
- [ ] Checked error code documentation?
- [ ] Asked colleague (fresh eyes help)?
- [ ] Tried simplest fix first?
- [ ] Reproduced in different environment?
- [ ] Examined recent deployments?
- [ ] Checked external service status?

### Tools to Learn

- Python profilers (cProfile, line_profiler, memory_profiler)
- Database query analyzer (EXPLAIN)
- Network debugging (tcpdump, Wireshark)
- Distributed tracing (Jaeger, DataDog)
- System monitoring (top, ps, netstat)
- Log aggregation (ELK Stack, Datadog, CloudWatch)

---

**Next Resources:**

- See DEPLOYMENT_AND_DEVOPS_GUIDE.ipynb for production runbooks
- See DISASTER_RECOVERY_AND_INCIDENT_MANAGEMENT.ipynb for incident response
